# Week 3: data interpretation and focused EDA
Completed October 16, 2026, for the October 16–18 work block.

Read [the technical review](docs/week3_review.md) for missing-value decisions, leakage, dependencies, source questions and the next implementation stage. Full column decisions are in [student_column_decisions.csv](outputs/week3/student_column_decisions.csv) and [tensile_column_decisions.csv](outputs/week3/tensile_column_decisions.csv).

Reproduce all review tables and figures with `python scripting/week3_eda.py` after installing `requirements-week3.txt`. The cells below read the generated evidence; they do not apply preprocessing or train models. The raw data remains unchanged. Allowed semester predictors are considered in the main retrospective classification task; an enrollment-only comparison is optional.


In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripting/week3_eda.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the DSFIP repository.')
OUT = ROOT / 'outputs/week3'
summary = json.loads((OUT / 'summary.json').read_text())
print('Raw students:', summary['student_raw_rows'])
print('Binary students:', summary['student_binary_rows'], summary['student_class_counts'])
print('Raw tensile records:', summary['tensile_raw_rows'])
print('Observed UTS:', summary['uts_observed_rows'], '; missing UTS:', summary['uts_missing_rows'])
print('Transformations applied:', summary['transformations_applied'])
print('Models trained:', summary['models_trained'])


Raw students: 4424
Binary students: 3630 {'Graduate': 2209, 'Dropout': 1421}
Raw tensile records: 1050
Observed UTS: 489 ; missing UTS: 561
Transformations applied: False
Models trained: False


In [2]:
student_columns = pd.read_csv(OUT / 'student_column_decisions.csv')
print(student_columns[['column','semantic_type','missing_count','coded_unknown_count']].to_string(index=False))
print('\nStudent code definitions: qualification 34 = Unknown; occupation 99 = blank; occupation 0 = Student.')
print('\nStrongest academic correlations (descriptive, not an automatic drop rule):')
print(pd.read_csv(OUT / 'student_semester_correlation_pairs.csv').head(8).to_string(index=False))
print('\nZero-grade recording:')
print(pd.read_csv(OUT / 'student_zero_grade_review.csv').to_string(index=False))


                                        column      semantic_type  missing_count  coded_unknown_count
                                Marital status        categorical              0                    0
                              Application mode        categorical              0                    0
                             Application order    ordinal numeric              0                    0
                                        Course        categorical              0                    0
                    Daytime/evening attendance binary categorical              0                    0
                        Previous qualification        categorical              0                    0
                Previous qualification (grade)            numeric              0                    0
                                   Nacionality        categorical              0                    0
                        Mother's qualification        categorical              0  

In [3]:
tensile_columns = pd.read_csv(OUT / 'tensile_column_decisions.csv')
print(tensile_columns[['column','role','missing_count','missing_percent','nonnumeric_observed_count']].to_string(index=False))
print('\nUnparsed numeric tokens (raw strings are preserved):')
print(pd.read_csv(OUT / 'numeric_token_review.csv').to_string(index=False))
print('\nGeometry dependencies:')
print(pd.read_csv(OUT / 'geometry_dependency_review.csv').to_string(index=False))
print('\nSource-row issues; flags are not confirmed corrections:')
print(pd.read_csv(OUT / 'quality_issues.csv').to_string(index=False))


                         column                                      role  missing_count  missing_percent  nonnumeric_observed_count
                      Reference group/audit metadata; forbidden predictor              0             0.00                          0
                       Material                         allowed predictor              0             0.00                          0
                 Material Group                         allowed predictor              0             0.00                          0
                       C (wt.%)                         allowed predictor             89            18.20                         22
                      Si (wt.%)                         allowed predictor            122            24.95                          0
                      Mn (wt.%)                         allowed predictor            120            24.54                          0
                       P (wt.%)                         allowed predi

In [4]:
files = [ROOT / 'data/raw/student_dropout/data.csv', ROOT / 'data/raw/tensile_strength/Tensile_Properties_of_Sub-sized_Specimens.xlsx']
for path in files:
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual == summary['source_sha256'][path.name], f'Source changed: {path.name}'
assert len(student_columns) == summary['student_raw_columns'] == 37
assert len(tensile_columns) == summary['tensile_raw_columns'] == 55
assert student_columns.column.is_unique and tensile_columns.column.is_unique
assert student_columns.meaning.notna().all() and tensile_columns.meaning.notna().all()
print('All 92 columns covered once; raw-data checksums match the executed review.')
print('No source measurements corrected, missing predictors filled, or outliers removed.')


All 92 columns covered once; raw-data checksums match the executed review.
No source measurements corrected, missing predictors filled, or outliers removed.


## Figures
All charts use the binary student or observed-UTS subset. The correlation tables include pairwise sample counts. Associations are descriptive and do not establish causation or individual feature availability before dropout. All extremes remain in the data.

![Class distribution and zero grades](outputs/week3/student_outcomes_and_zero_grades.png)
![Semester relationships](outputs/week3/student_semester_correlations.png)
![Missing predictor coverage](outputs/week3/tensile_predictor_missingness.png)
![UTS distribution by material](outputs/week3/tensile_uts_by_material.png)
![Tensile numerical relationships](outputs/week3/tensile_numeric_correlations.png)
![Source-study reporting patterns](outputs/week3/tensile_missingness_by_reference.png)

## Open decisions
See the review document for exact source-row follow-ups. Geometry, excessive composition totals, student count definitions and range/limit/balance conversions remain source questions; no correction is invented. The four-person roster is confirmed: Mallory Roussel, Noah Torres, Greyson Alegria and Shane Roth. Named model responsibilities remain to be assigned. Actual member contributions and remote GitHub privacy have not been verified.

Next stage: implement justified, training-fitted preprocessing and fixed evaluation splits. First model training follows that work. Remote publication is separate from the local review.
